In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Exploratory Data Analysis

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("=== DATA SHAPE & COLUMNS ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")

print("=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== SAMPLE HEAD ===")
print(df.head(2))

if 'answer' in df.columns:
    print("\n=== TARGET DISTRIBUTION ===")
    print(df['answer'].value_counts(dropna=False, normalize=True) * 100)

In [ ]:
option_cols = ['A', 'B', 'C', 'D', 'E']
prompt_col = 'prompt'
answer_col = 'answer'

df['answer_col'] = df.apply(lambda row: row[row['answer']], axis=1)

if prompt_col:
    df['prompt_word_len'] = df[prompt_col].astype(str).apply(lambda x: len(x.split()))
if answer_col:
    df['answer_word_len'] = df['answer_col'].astype(str).apply(lambda x: len(x.split()))

for col in option_cols:
    df[f'{col}_word_len'] = df[col].astype(str).apply(lambda x: len(x.split()))

df['total_word_len'] = df[[c for c in df.columns if 'word_len' in c]].sum(axis=1)

print("=== LENGTH STATS (WORD COUNTS) ===")
print(df[[c for c in df.columns if 'word_len' in c]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
for col in option_cols:
    df[f'{col}_char_len'] = df[col].astype(str).apply(len)

if 'answer' in df.columns:
    def get_correct_option_len(row):
        ans = str(row['answer']).strip()
        for col in option_cols:
            if ans == col:
                return len(str(row[col]))
        return None

    df['correct_option_char_len'] = df.apply(get_correct_option_len, axis=1)
    
    print("=== OPTION LENGTH COMPARISON ===")
    print(f"Mean Correct Option Length: {df['correct_option_char_len'].mean():.2f} chars")
    print("Mean Option Lengths by Column:")
    for col in option_cols:
        print(f"  {col}: {df[f'{col}_char_len'].mean():.2f} chars")

In [ ]:
if prompt_col:
    exact_dup_prompts = df[prompt_col].duplicated().sum()
    print("=== DUPLICATE ANALYSIS ===")
    print(f"Exact Duplicate Prompts: {exact_dup_prompts} ({(exact_dup_prompts/len(df))*100:.2f}%)")

def check_duplicate_options(row):
    opts = [str(row[col]).strip() for col in option_cols]
    return len(opts) != len(set(opts))

df['has_duplicate_options'] = df.apply(check_duplicate_options, axis=1)
print(f"Questions with duplicate options: {df['has_duplicate_options'].sum()}")

In [ ]:
import re

def detect_patterns(text):
    text = str(text)
    has_latex = bool(re.search(r'\$|\\frac|\\sqrt|\\alpha|\\beta', text))
    has_code = bool(re.search(r'def\s+|class\s+|import\s+|for\s+.*in|#include', text))
    has_math_symbols = bool(re.search(r'[=+\-*/^<>≤≥≠±]', text))
    return pd.Series([has_latex, has_code, has_math_symbols], index=['has_latex', 'has_code', 'has_math_symbols'])

if prompt_col:
    pattern_df = df[prompt_col].apply(detect_patterns)
    print("=== DOMAIN & SYNTAX PATTERNS ===")
    print(pattern_df.mean() * 100)

Checking Models availability on Hugging face.

# Roberta Base Model ( Pretrained )

In [ ]:
import transformers
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

print(f"=== TRANSFORMERS VERSION ===")
print(f"Version: {transformers.__version__}\n")

candidates = [
    "roberta-base",
    "roberta-large",
    "microsoft/deberta-v3-base",
    "microsoft/deberta-v3-small",
    "bert-base-uncased",
    "google/electra-base-discriminator"
]

print("=== CHECKING HF MODEL ACCESSIBILITY ===")
user_secrets = UserSecretsClient()
api = HfApi(token=user_secrets.get_secret("HF_KEY"))

for model_id in candidates:
    try:
        info = api.model_info(model_id)
        print(f"✅ FOUND: '{model_id}'")
    except Exception as e:
        print(f"❌ NOT FOUND / ERROR: '{model_id}' -> {e}")

## Initialization

In [ ]:
import os
import torch
import wandb
from dataclasses import dataclass
from typing import Optional, Union
from transformers import (
AutoTokenizer, 
AutoModelForMultipleChoice, 
TrainingArguments, 
Trainer,
PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

# 1. automatic W&B authentication
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("WB_KEY")
# wandb.login(key=secret_value_0)

# wandb.init(
#     project="23f2002267-t22026", 
#     name="roberta-base-baseline",
#     reinit="finish_previous",
#     #config="{}",
#     #notes="",
#     #dir="./wandb_logs",
#     #resume="allow, must, never",
# )

MODEL_NAME = "roberta-base"
MAX_LEN = 256
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [ ]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

## Data collator and functions

In [ ]:
def preprocess_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']] # copies for options
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

In [ ]:
def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

In [ ]:
from datasets import Dataset
gkf = GroupKFold(n_splits=5)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df = df[df['fold'] == 0].reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(preprocess_function, batched=True)

In [ ]:
#previous underperforming parameters:
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     warmup_steps=50,                  # Updated from warmup_ratio to avoid deprecation warning
#     max_grad_norm=1.0,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=2,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_steps=20,
#     report_to="wandb",
#     load_best_model_at_end=True,
#     metric_for_best_model="accuracy",
#     fp16=torch.cuda.is_available()
# )

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
training_args = TrainingArguments(
        output_dir=f"./results_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",
        save_only_model=True,
        learning_rate=1.5e-5,             # 1. Lowered LR for fine-grained convergence
        warmup_steps=100,                 # Smooth warmup over the first ~10% of total steps
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,    # Effective batch size = 8
        num_train_epochs=4,               # 2. Increased to 4 epochs
        label_smoothing_factor=0.05,      # 3. Prevents overconfidence, boosts MAP@3 ranking
        weight_decay=0.01,
        logging_steps=20,
        # report_to="wandb",
        fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
# wandb.finish()

## Test Data Inference and Submission

In [ ]:
from torch.utils.data import DataLoader

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(f"Test Set Shape: {test_df.shape}")
print(f"Sample Submission Columns: {list(sample_sub.columns)}")

# def preprocess_test_function(examples):
#     first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
#     second_sentences = [
#         [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
#         for i in range(len(examples['prompt']))
#     ]

#     first_sentences = sum(first_sentences, [])
#     second_sentences = sum(second_sentences, [])

#     tokenized_examples = tokenizer(
#         first_sentences, 
#         second_sentences, 
#         truncation=True, 
#         max_length=MAX_LEN,
#         padding=False
#     )

#     return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

class TestDataCollatorForMultipleChoice:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_input_cols} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=True,
            return_tensors="pt",
        )

        return {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=TestDataCollatorForMultipleChoice(tokenizer=tokenizer)
)

all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)  # Shape: (500, 5)

id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]

top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

print("\n=== TOP PREDICTION PATTERNS ===")
print(submission_df[target_col_name].value_counts().head(10))

# Ensemble version (5 - folds):

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import wandb
from dataclasses import dataclass
from typing import Optional, Union
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer,
    PreTrainedTokenizerBase
)
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient


user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

## Initilization

In [ ]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256
NUM_FOLDS = 5
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df['label'] = df['answer'].map(label2id)

gkf = GroupKFold(n_splits=NUM_FOLDS)
df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=df['prompt'])):
    df.loc[val_idx, 'fold'] = fold

oof_predictions = np.zeros((len(df), 5))

## Functions and Data collator

In [ ]:
def preprocess_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}
    
class TrainDataCollatorForMultipleChoice:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        labels = [feature.pop("label") for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        model_inputs = ["input_ids", "attention_mask", "token_type_ids"]
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_inputs} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,

            padding=True,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

## Preporcessing and Training

In [ ]:
for fold in range(NUM_FOLDS):
    print(f"\n========================== TRAINING FOLD {fold + 1}/{NUM_FOLDS} ==========================")
    
    wandb.init(
        project="23f2002267-t22026", 
        name=f"roberta-base-fold{fold}",
        reinit="finish_previous"
    )

    fold_train = df[df['fold'] != fold].reset_index(drop=True)
    fold_val = df[df['fold'] == fold].reset_index(drop=True)

    train_dataset = Dataset.from_pandas(fold_train).map(preprocess_function, batched=True)
    val_dataset = Dataset.from_pandas(fold_val).map(preprocess_function, batched=True)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    training_args = TrainingArguments(
        output_dir=f"./results_fold{fold}",
        eval_strategy="epoch",
        save_strategy="no",               # FIXED: Disable intermediate checkpoint saving to prevent disk full error
        save_only_model=True,             # FIXED: Save only lightweight model weights
        learning_rate=2e-5,
        warmup_steps=50,
        max_grad_norm=1.0,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_steps=20,
        report_to="wandb",
        fp16=torch.cuda.is_available()
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer, 
        data_collator=TrainDataCollatorForMultipleChoice(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    # Save lightweight final model for this fold
    model.save_pretrained(f"./model_fold_{fold}")
    tokenizer.save_pretrained(f"./model_fold_{fold}")
    
    # Clean GPU memory between folds
    del model, trainer
    torch.cuda.empty_cache()
    wandb.finish()

## Test data inference ans  Submission

In [ ]:
print("\n========================== STARTING 5-FOLD ENSEMBLE INFERENCE ==========================")

test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

class TestDataCollatorForMultipleChoice:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_input_cols} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=True,
            return_tensors="pt",
        )

        return {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}

test_loader = DataLoader(
    test_dataset, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=TestDataCollatorForMultipleChoice(tokenizer=tokenizer)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_fold_logits = []

# Accumulate test predictions across all 5 trained models
for fold in range(NUM_FOLDS):
    print(f"Running inference with Fold {fold} model...")
    fold_model = AutoModelForMultipleChoice.from_pretrained(f"./model_fold_{fold}")
    fold_model.eval()
    fold_model.to(device)

    fold_logits = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            outputs = fold_model(input_ids=input_ids, attention_mask=attention_mask)
            fold_logits.append(outputs.logits.cpu().numpy())

    all_fold_logits.append(np.concatenate(fold_logits, axis=0))
    del fold_model
    torch.cuda.empty_cache()

# Average logits across all 5 folds
mean_logits = np.mean(all_fold_logits, axis=0)  # Shape: (500, 5)

# Extract Top-3 MAP@3 Choices
top3_indices = np.argsort(mean_logits, axis=1)[:, ::-1][:, :3]
top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

# Write final ensembled submission CSV
id_col_name = sample_sub.columns[0]
target_col_name = sample_sub.columns[1]

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== ENSEMBLED MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

## MLP-Scratch model

## Imports and Config

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wandb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

# 1. W&B Initialization
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="MLP-Scratch-v10",
    reinit="finish_previous",
    config={
        "MAX_FEATURES": 5000,
        "batch_size": 16,
        "hidden1": 256,
        "BatchNorm1d":256,
        "dropout": 0.3,
        "hidden2": 64,
        "BatchNorm1d": 64,
        "dropout": 0.2,
        "epochs": 10
    }
)

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
train_df['label'] = train_df['answer'].map(label2id)

gkf = GroupKFold(n_splits=5)
train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['prompt'])):
    train_df.loc[val_idx, 'fold'] = fold

train_split = train_df[train_df['fold'] != 0].reset_index(drop=True)
val_split = train_df[train_df['fold'] == 0].reset_index(drop=True)

option_cols = ['A', 'B', 'C', 'D', 'E']

corpus = []
for _, row in train_split.iterrows():
    corpus.append(str(row['prompt']))
    for opt in option_cols:
        corpus.append(str(row[opt]))


MAX_FEATURES = 5000
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, stop_words='english')
vectorizer.fit(corpus)

## Functions

In [ ]:
def extract_tfidf_features(df):
    X = []
    y = []
    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([str(row['prompt'])]).toarray()[0]
        row_opts = []
        for opt in option_cols:
            opt_vec = vectorizer.transform([str(row[opt])]).toarray()[0]
            # # Absolute difference feature vector
            # diff_vec = np.abs(prompt_vec - opt_vec)
            # row_opts.append(diff_vec)
            # Concatenate Prompt TF-IDF vector and Option TF-IDF vector
            combined_vec = np.concatenate([prompt_vec, opt_vec])
            row_opts.append(combined_vec)
        X.append(row_opts)
        y.append(row['label'])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

## Data Prep and loading

In [ ]:
print("Extracting TF-IDF Features for Train & Val...")
X_train, y_train = extract_tfidf_features(train_split)  # Shape: (1600, 5, 10000)
X_val, y_val = extract_tfidf_features(val_split)        # Shape: (400, 5, 10000)

class MCQDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(MCQDataset(X_train, y_train), batch_size=16, shuffle=True) #16
val_loader = DataLoader(MCQDataset(X_val, y_val), batch_size=16, shuffle=False) #16

## MLP config

In [ ]:
class DeepMLPMCQSolver(nn.Module):
    def __init__(self, input_dim):
        super(DeepMLPMCQSolver, self).__init__()
        # 3-Layer Feedforward Neural Network
        self.scoring_head = nn.Sequential(
            nn.Linear(input_dim, 256), #256
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3), #0.3
            
            nn.Linear(256, 64), #256, 64
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 1)  # 64, 1
        )

    def forward(self, x):
        # Input shape: (batch_size, 5, input_dim)
        batch_size, num_options, feature_dim = x.shape
        x_flat = x.view(-1, feature_dim)  # Flatten to (batch_size * 5, feature_dim)
        
        logits = self.scoring_head(x_flat) # Shape: (batch_size * 5, 1)
        logits = logits.view(batch_size, num_options)  # Unflatten to (batch_size, 5)
        return logits

## Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[2]
model = DeepMLPMCQSolver(input_dim=input_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2) #weight_decay=1e-4

# 7. Training Loop
EPOCHS = 10 # 15
print(f"Starting Training for {EPOCHS} Epochs on {device}...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(batch_y)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == batch_y).sum().item()
        total += len(batch_y)
        
    train_acc = correct / total
    avg_train_loss = train_loss / total
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            
            val_loss += loss.item() * len(batch_y)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            
    val_acc = val_correct / val_total
    avg_val_loss = val_loss / val_total
    
    # Log metrics to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

wandb.finish()

## Test Data Inference and Submission

In [ ]:
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(f"Test Set Shape: {test_df.shape}")

# 2. Feature Extraction for Test Set using the Trained TF-IDF Vectorizer
def extract_test_features(df):
    X = []
    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([str(row['prompt'])]).toarray()[0]
        row_opts = []
        for opt in option_cols:
            opt_vec = vectorizer.transform([str(row[opt])]).toarray()[0]
            # Concatenate Prompt TF-IDF vector and Option TF-IDF vector
            combined_vec = np.concatenate([prompt_vec, opt_vec])
            row_opts.append(combined_vec)
        X.append(row_opts)
    return np.array(X, dtype=np.float32)

print("Extracting TF-IDF Features for Test Set...")
X_test = extract_test_features(test_df)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

# 3. Model Inference
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor).cpu().numpy()  # Shape: (500, 5)

top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]

top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

# 5. Format and Save submission.csv
id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== SUBMISSION HEAD ===")
print(submission_df.head(10))

print("\n=== PREDICTION DISTRIBUTION ===")
print(submission_df[target_col_name].value_counts().head(10))

    MLP-Scratch-baseline : 0.60390
    MLP-Scratch-v1 : 0.61512
    MLP-Scratch-v3 : 0.70532
    MLP-Scratch-v4 : 0.70656
    MLP-Scratch-v5 : 0.71197
    MLP-Scratch-v6 : 0.70989
    MLP-Scratch-v7 : 0.70822
    MLP-Scratch-v8 : 0.70448
    MLP-Scratch-v9 : 0.70365
    MLP-Scratch-v10 : 0.70989